# 05 — Reactive Observers

Agents are passive until an observer makes them reactive. lionag2 uses AG2's `@agent.observer(EventType)` to wire reactive coordination:

- When a `FindingEmitted` with high novelty arrives → observer spawns a child team at depth+1.
- When a `PaperGapEvent` fires → observer triggers depth expansion to fill the gap.
- When a `ContradictionFound` arrives → observer records it for the cross-check.

Observers can be **sync or async**. They fire every time the agent emits a matching event type.

In [ ]:
from autogen.beta import Agent, MemoryStream
from autogen.beta.config import OpenAIConfig
from autogen.beta.events import BaseEvent

from lionag2.research.events import DepthRequested, FindingEmitted

## Observer — react to events

An `@agent.observer(FindingEmitted)` fires its handler each time the agent emits a `FindingEmitted`.

In [ ]:
import os

config = OpenAIConfig("gpt-5.4-mini", api_key=os.getenv("OPENAI_API_KEY"))

captured: list[FindingEmitted] = []

agent = Agent("watcher", prompt="You observe findings.", config=config)


@agent.observer(FindingEmitted)
def on_finding(event: FindingEmitted) -> None:
    captured.append(event)
    print(f"  Observer fired: novelty={event.novelty:.2f} claim={event.claim[:50]}")


print("Observer registered. It will fire when the agent emits FindingEmitted events.")
print(f"Captured so far: {len(captured)}")

## The reactive chain

In the engine, observers on each agent check novelty against a threshold. High-novelty findings trigger depth expansion — spawning a child team at depth+1.

```python
@agent.observer(FindingEmitted)
def _on_finding(event: FindingEmitted) -> None:
    if event.novelty >= self.novelty_threshold:
        self._spawn(self._spawn_depth_node(
            event.claim, parent_node_id=node_id, depth=depth + 1
        ))
```

The research tree **emerges from observer reactions**, not from an imperative BFS loop.

In [ ]:
# In the real engine, the reactive chain looks like:
#
# FindingEmitted(novelty=0.9)
#   → observer on agent fires
#   → engine._spawn_depth_node(claim, depth+1)
#   → child team runs concurrently
#   → child emits its own findings
#   → observers on child agents fire
#   → grandchildren spawn (if under max_depth)
#
# No explicit bus routing needed — observers are wired
# directly on each agent at creation time.

print("Reactive chain: FindingEmitted → observer → spawn child")
print("Each agent gets observers wired in _make_agent()")

## Depth expansion control

The engine controls depth expansion with:

- **`novelty_threshold`** (default 0.7) — only findings above this trigger children.
- **`max_depth`** — caps the recursion tree.
- **`max_concurrent`** — limits concurrent depth nodes.
- **Topic deduplication** — `_seen_topics` prevents re-investigating the same question.

## All observers in the engine

Each agent created by `_make_agent()` gets these observers:

```python
@agent.observer(FindingEmitted)
def _on_finding(event):      # novelty → depth expansion

@agent.observer(DepthRequested)
def _on_depth(event):        # explicit depth request → spawn child

@agent.observer(ContradictionFound)
def _on_contradiction(event): # record for cross-check

@agent.observer(PivotDetected)
def _on_pivot(event):        # record for cross-check

@agent.observer(HandoffRequested)
def _on_handoff(event):      # agent-to-agent routing

@agent.observer(ToolResultsEvent)
def _on_tools(event):        # URL capture from Exa
```

The paper writer gets an additional `PaperGapEvent` observer for gap→depth feedback.

## Up next

Agents need bounded context to stay effective over long conversations. Tutorial 06 introduces assembly policies and the knowledge store.